In [ ]:
# --- 1. Installation of Required Libraries ---

print("Installing libraries: mediapipe, opencv-python, scikit-learn, tensorflow, tqdm...")
!pip install -q mediapipe opencv-python scikit-learn tensorflow tqdm
print("Installation complete.")


Installing libraries: mediapipe, opencv-python, scikit-learn, tensorflow, tqdm...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
Installation complete.


In [ ]:
!pip install --upgrade numpy protobuf
!pip show mediapipe

Name: mediapipe
Version: 0.10.21
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: /usr/local/lib/python3.11/dist-packages
Requires: absl-py, attrs, flatbuffers, jax, jaxlib, matplotlib, numpy, opencv-contrib-python, protobuf, sentencepiece, sounddevice
Required-by: 


In [ ]:
# --- 2. Import Libraries and Mount Google Drive ---
import os
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from google.colab import drive

In [ ]:
# --- 3. Mount Drive ---
print("\nMounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted successfully.")


Mounting Google Drive...
Mounted at /content/drive
Drive mounted successfully.


In [ ]:
# --- 4. Configuration and Setup ---

# PATH YOUR GOOGLE DRIVE.
DATASET_BASE_PATH = "/content/drive/MyDrive/major_project/dataset/original_increased_sampled_mixture_augmentation"

# fixed length for each video sequence.
SEQ_LEN = 75

print("\nInitializing MediaPipe Holistic...")
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False, min_detection_confidence=0.5, min_tracking_confidence=0.5)


Initializing MediaPipe Holistic...


In [ ]:
# --- 5. Landmark and Sequence Processing Functions (start)-----------------------------------------------------(start 75)

# Define the specific pose landmarks we want to extract.
DESIRED_POSE_INDICES = [0, 1, 4, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]

# Calculate the total number of features (x, y, z) per frame.
NUM_FEATURES_PER_FRAME = len(DESIRED_POSE_INDICES) * 3 + 21 * 3 + 21 * 3

# total 177 features

def extract_keypoints(results):
    """
    Extracts keypoints for a custom set of pose landmarks and all hand landmarks.
    """
    keypoints = []
    # Extract Pose landmarks (x, y, z only)
    if results.pose_landmarks:
        for i in DESIRED_POSE_INDICES:
            lm = results.pose_landmarks.landmark[i]
            keypoints.extend([lm.x, lm.y, lm.z])
    else:
        keypoints.extend([0.0] * len(DESIRED_POSE_INDICES) * 3)

    # Extract Left Hand landmarks
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            keypoints.extend([lm.x, lm.y, lm.z])
    else:
        keypoints.extend([0.0] * 21 * 3)

    # Extract Right Hand landmarks
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            keypoints.extend([lm.x, lm.y, lm.z])
    else:
        keypoints.extend([0.0] * 21 * 3)

    return keypoints

def process_video(video_path):
    """
    Processes a single video file to extract a sequence of keypoints from all its frames.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Warning: Could not open video file: {video_path}")
        return []

    frame_sequence = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Process frame with MediaPipe and extract keypoints
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(frame_rgb)
        keypoints = extract_keypoints(results)
        frame_sequence.append(keypoints)

    cap.release()
    return np.array(frame_sequence)

def standardize_sequence(sequence, target_len):
    """
    Standardizes a sequence to a target length by either truncating the end
    or padding with zeros.
    """
    current_len = len(sequence)

    if current_len > target_len:
        # Take the sequence from the start
        return sequence[:target_len]
    elif current_len < target_len:
        # Pad the sequence if it's too short
        padding_shape = (target_len - current_len, sequence.shape[1])
        padding = np.zeros(padding_shape)
        return np.vstack((sequence, padding))
    else:
        # Return as is if the length is correct
        return sequence

In [ ]:
# --- 2. Main Data Loading and Processing Loop ---

print("\nStarting data loading and processing from Google Drive...")
classes = sorted([d for d in os.listdir(DATASET_BASE_PATH) if os.path.isdir(os.path.join(DATASET_BASE_PATH, d))])
print(f"Detected classes: {classes}")

features = []
labels_raw = []

VIDEO_EXTENSIONS = (".mp4", ".mov", ".avi", ".mkv", ".webm", ".wmv")

for label in tqdm(classes, desc="Processing Classes"):
    class_dir = os.path.join(DATASET_BASE_PATH, label)
    video_files = [f for f in os.listdir(class_dir) if f.lower().endswith(VIDEO_EXTENSIONS)]

    for file_name in tqdm(video_files, desc=f"  Videos in {label}", leave=False):
        file_path = os.path.join(class_dir, file_name)
        raw_sequence = process_video(file_path)

        if raw_sequence.size == 0 or (raw_sequence.ndim > 1 and raw_sequence.shape[1] != NUM_FEATURES_PER_FRAME):
            print(f"Skipping malformed or empty video: {file_path}")
            continue

        standardized_seq = standardize_sequence(raw_sequence, SEQ_LEN)
        features.append(standardized_seq)
        labels_raw.append(label)

X = np.array(features)

le = LabelEncoder()
y_encoded = le.fit_transform(labels_raw)
y = to_categorical(y_encoded)

print("\n--- Processing Complete ---")
print(f"Shape of feature data (X): {X.shape}")
print(f"Shape of label data (y): {y.shape}")
print(f"Feature shape is (num_samples, sequence_length, num_features_per_frame).")
print(f"Classes mapping: {list(zip(le.classes_, range(len(le.classes_))))}")


Starting data loading and processing from Google Drive...
Detected classes: ['Afternoon', 'Apple', 'April', 'August', 'Banana', 'Day', 'December', 'Evening', 'Febraury', 'Friday', 'Grapes', 'January', 'July', 'June', 'March', 'May', 'Monday', 'Morning', 'Night', 'November', 'October', 'Orange', 'Rainy', 'Saturday', 'September', 'Summer', 'Sunday', 'Thursday', 'Tuesday', 'Valencia_Orange', 'Watermelon', 'Wednesday', 'Winter']


Processing Classes: 100%|██████████| 33/33 [5:54:15<00:00, 644.11s/it]



--- Processing Complete ---
Shape of feature data (X): (3960, 75, 177)
Shape of label data (y): (3960, 33)
Feature shape is (num_samples, sequence_length, num_features_per_frame).
Classes mapping: [(np.str_('Afternoon'), 0), (np.str_('Apple'), 1), (np.str_('April'), 2), (np.str_('August'), 3), (np.str_('Banana'), 4), (np.str_('Day'), 5), (np.str_('December'), 6), (np.str_('Evening'), 7), (np.str_('Febraury'), 8), (np.str_('Friday'), 9), (np.str_('Grapes'), 10), (np.str_('January'), 11), (np.str_('July'), 12), (np.str_('June'), 13), (np.str_('March'), 14), (np.str_('May'), 15), (np.str_('Monday'), 16), (np.str_('Morning'), 17), (np.str_('Night'), 18), (np.str_('November'), 19), (np.str_('October'), 20), (np.str_('Orange'), 21), (np.str_('Rainy'), 22), (np.str_('Saturday'), 23), (np.str_('September'), 24), (np.str_('Summer'), 25), (np.str_('Sunday'), 26), (np.str_('Thursday'), 27), (np.str_('Tuesday'), 28), (np.str_('Valencia_Orange'), 29), (np.str_('Watermelon'), 30), (np.str_('Wednesd

In [ ]:
# --- Define Save Path and Save Files ---

SAVE_PATH = '/content/drive/MyDrive/major_project/features'
os.makedirs(SAVE_PATH, exist_ok=True)

# Define file paths for features and labels
X_save_path = os.path.join(SAVE_PATH, 'X_original_increased_sampled_mixture_augmentation.npy')
y_save_path = os.path.join(SAVE_PATH, 'y_original_increased_sampled_mixture_augmentation.npy')

# Save the arrays using NumPy
print(f"\nSaving feature data to: {X_save_path}")
np.save(X_save_path, X)

print(f"Saving label data to: {y_save_path}")
np.save(y_save_path, y)

print("\n--- Data Saved ---")
print("You can now find X_original_increased_sampled_mixture_augmentation.npy and y_original_increased_sampled_mixture_augmentation.npy in your Google Drive folder.")


Saving feature data to: /content/drive/MyDrive/major_project/features/X_original_increased_sampled_mixture_augmentation.npy
Saving label data to: /content/drive/MyDrive/major_project/features/y_original_increased_sampled_mixture_augmentation.npy

--- Data Saved ---
You can now find X_original_increased_sampled_mixture_augmentation.npy and y_original_increased_sampled_mixture_augmentation.npy in your Google Drive folder.
